In [1]:
import ast
import os
import optuna
import ipynbname
import pandas as pd
import numpy as np
from optuna.exceptions import TrialPruned
from Functions.AutoCloud_V4_5 import *
from Functions.TedaOptimize_V4_5 import *
from Functions.Utils import *
from Functions.Graphs import *
from Functions.TedaGraphs import *
from Functions.Utils_OPT_R2 import *
import copy

FileName = ipynbname.name()

# Configuração do experimento
FEEDBACK = 'symmetric'    # 'symmetric' (gradiente no passo atual) ou 'random' (ablação: RFLO + DFA)
WTAG, WTAP = False, False # configuração principal: soma ponderada dos grânulos (eq. 36)
EOL = 0.3                 # limiar de fim de vida (EoL) do HI
W = 1                       # janela do DSI em cada X_k (nº de amostras do RS; PrepareData/WindowDSI)

# rho dinâmico (AutoCloud_V4_2): começa em 0 e é ajustado a cada iteração maximizando V = média(cov)*média(sp)
RHO_TARGET = 'RUL'           # 'HI' (causal, HI um passo à frente) ou 'RUL' (oráculo: usa a RUL real; n_rho previsões por passo)
RHO_MIN, RHO_MAX, N_RHO = 1e-3, 1, 4   # grade log-espaçada de candidatos
LAM = 1.0                   # fator de esquecimento (1.0 = todos os passos com o mesmo peso)

out_path = os.path.join('Optimization', RunTag(FileName,FEEDBACK,WTAG,WTAP), 'multi', 'Optimization.csv')
dfs = [pd.read_csv(out_path)] if os.path.exists(out_path) else []
RS = pd.read_excel(r'Dataset\RS.xlsx')
HI = pd.read_excel(r'Dataset\HI.xlsx')

sig = HI['PC1'].values

c:\Users\Usuario\Desktop\Python\Batteries_Prognostics\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
params = [3.5,28,[70],8,0,1,2,5,19,0]
params = [2.5,33,[32, 49],1,2,8,9,5,2,1,1]
m,nI,nR,nO,mO,N1,N2,N3,τ,act = ParseParams(params)
X, Y, Z, kX = PrepareData(RS, HI, W, nI, nO, mO)
ref, J, L = RefEoL(sig, EOL, kX)   # avisa se o fluxo não alcança o EoL
if isinstance(nR, str): nR = ast.literal_eval(nR)

RHO_DYN, RHO_ETA, RHO_T, RHO_BETA = True, 0.1, 0.5, 0.75              # False: rho estático (= RHO), passo, temperatura da cobertura suavizada (escore |e|/e_est), momento
RHO, RHO_MIN, RHO_MAX = 0.5, 1e-2, 10
INTR_STATE = 'reset'       

teda=AutoCloud(m=m,nI=nI,nR=nR,nO=nO,ηS=[N1,N2,N3],act=act,
               tau=τ,eol=EOL,ref=ref,wtaG=WTAG,wtaP=WTAP,feedback=FEEDBACK,mO=mO,
               rho=RHO,lam=LAM,intr_state=INTR_STATE,
               rho_dyn=RHO_DYN,rho_eta=RHO_ETA,rho_T=RHO_T,rho_beta=RHO_BETA,rho_min=RHO_MIN,rho_max=RHO_MAX,
               rls_mode='pol',nf_rls=3,ff_rls=0.9,dt_rls=1e3,
               )

teda.SimGranularRUL(J,L,X,Y,Z)

print(f"HI (HI_Granularity): cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f} | Vsum={teda.Vsum:.4f}")
PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=3,dsi_mode='hide')


HI (HI_Granularity): cov=0.346 | sp=0.756 | V=média(cov*sp)=0.2780 | Vsum=22.5156


In [4]:
brngs = [f'B1_{i+1}' for i in range(5)]
brngs = ['B1_1','B1_2','B1_3','B2_2','B2_4',]
paths = [f'Dataset\DSIR1n\{brng}.csv' for brng in brngs]
DSIs = [pd.read_csv(path) for path in paths]
DSI1, DSI2, DSI3, DSI4, DSI5 = DSIs

teda=AutoCloud(m=2.0,nI=nI,nR=nR,nO=nO,ηS=[N1,N2,N3],act=act,
               tau=τ,eol=EOL,ref=ref,wtaG=WTAG,wtaP=WTAP,feedback=FEEDBACK,mO=mO,
               rho=RHO,lam=LAM,intr_state=INTR_STATE,
               rho_dyn=RHO_DYN,rho_eta=RHO_ETA,rho_T=RHO_T,rho_beta=RHO_BETA,rho_min=RHO_MIN,rho_max=RHO_MAX,
               rls_mode='pol',nf_rls=3,ff_rls=0.9,dt_rls=1e3,
               )
w = 3
teda.SimGranulation(WindowDSI(DSI1,w))   
teda.SimGranulation(WindowDSI(DSI2,w))   
teda.SimGranulation(WindowDSI(DSI3,w))   
teda.SimGranulation(WindowDSI(DSI4,w))   
teda.SimGranulation(WindowDSI(DSI5,w))   

print(f"HI (HI_Granularity): cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f} | Vsum={teda.Vsum:.4f}")
PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=5,dsi_mode='hide',)

HI (HI_Granularity): cov=0.000 | sp=0.000 | V=média(cov*sp)=0.0000 | Vsum=0.0000


In [8]:
dfRho

,V,rho,eta,T,beta,nf_rls,ff_rls,dt_rls
0,0.767459,1.978374,0.014830,0.085553,0.370355,3,0.861565,462629.510587
1,0.766795,1.357711,0.071528,0.114902,0.266767,5,0.861955,5406.908243
2,0.765743,0.643663,0.203220,0.075946,0.521248,5,0.852261,452839.533126
3,0.759849,2.112736,0.001889,0.547975,0.824740,4,0.839562,1035.504014
4,0.757084,2.028644,0.002699,0.353647,0.644665,6,0.870116,381.120377


In [14]:
path = r'Dataset\DSIR4\Bearing1_1.csv'
RS = pd.read_csv(path).iloc[:,-6:-1]
RS = dfMovingAvg(RS,10)
RS = dfExpMovAvg(RS,0.3)
RS = normalize_df(RS,0,25)
path = r'Dataset\HI4\Bearing1_1.csv'
HI = pd.read_csv(path)

sig = HI['PC1'].values

params = [3.5,19,[19, 54, 2, 56, 25, 42],8,7,1,3,2,15,0]
m,nI,nR,nO,mO,N1,N2,N3,τ,act = ParseParams(params)
X, Y, Z, kX = PrepareData(RS, HI, W, nI, nO, mO)
ref, J, L = RefEoL(sig, EOL, kX)   # avisa se o fluxo não alcança o EoL
if isinstance(nR, str): nR = ast.literal_eval(nR)

nf_rls, ff_rls, dt_rls = 3,0.861565,462629.510587
RHO, RHO_ETA, RHO_T, RHO_BETA = 1.978374, 0.014830, 0.085553, 0.370355
RHO_DYN, RHO_MIN, RHO_MAX = True, 1e-2, 10
INTR_STATE = 'carry'       

teda=AutoCloud(m=m,nI=nI,nR=nR,nO=nO,ηS=[N1,N2,N3],act=act,
               tau=τ,eol=EOL,ref=ref,wtaG=WTAG,wtaP=WTAP,feedback=FEEDBACK,mO=mO,
               rho=RHO,lam=LAM,intr_state=INTR_STATE,
               rho_dyn=RHO_DYN,rho_eta=RHO_ETA,rho_T=RHO_T,rho_beta=RHO_BETA,rho_min=RHO_MIN,rho_max=RHO_MAX,
               rls_mode='pol',nf_rls=nf_rls,ff_rls=ff_rls,dt_rls=dt_rls,
               )

teda.SimGranularRUL(J,L,X,Y,Z)

print(f"HI (HI_Granularity): cov={teda.cov:.3f} | sp={teda.sp:.3f} | V=média(cov*sp)={teda.V:.4f} | Vsum={teda.Vsum:.4f}")
PlotSimulation(teda=teda,mode='ply',xi=10, yf=100,ftrs=3,dsi_mode='hide')


HI (HI_Granularity): cov=0.947 | sp=0.808 | V=média(cov*sp)=0.7675 | Vsum=72.9086


In [6]:
params = [3.5,19,[19, 54, 2, 56, 25, 42],8,7,1,3,2,15,0]
dfRho = OptimizeRho(
        RS=RS, HI=HI,
        FileName=None,params=params,OptSampler='random',OptSeed=None,
        n_study=1,timeout=None,n_trials=50,patience=None,
        rhoS=None,etaS=None,TS=None,betaS=None,nfS=None,ffS=None,dtS=None,
        rho_min=1e-2,rho_max=10,intr_state='reset',Vtarget='HI',rls_mode='pol',
        feedback='symmetric',wtaG=False,wtaP=False,
        eol=0.3,w=1,lam=1.0,
        OptReset=False,OptWarm=False,n_best=5)

OPTIMIZATION PROGRESS
Layers (nL): 6  |  Session: 1/1 (rho_HI_pol_915bdebe_s1)  |  Sampler: Random  |  Config: None_symmetric_G0P0
Trials in this study: 92 stored before this run  |  This run: 47/50  |  Elapsed: 00:01:55 / 00:10:00

BEST TRIAL
Trial #113 | Valor: 0.0137 | Params: {'rho': 1.9783744101505045, 'eta': 0.014829852961278442, 'T': 0.0855525589767045, 'beta': 0.37035516621002806, 'nf_rls': 3, 'ff_rls': 0.8615649125758276, 'dt_rls': 462629.51058691525}

LAST 3 TRIALS
Trial #136 [COMPLETE] | Score: 0.0142 | Params: {'rho': 1.3128773828324505, 'eta': 0.3865951971792505, 'T': 0.30782400520553743, 'beta': 0.4990129215366379, 'nf_rls': 4, 'ff_rls': 0.8136987276807166, 'dt_rls': 123204.49388665184}
Trial #137 [COMPLETE] | Score: 0.0155 | Params: {'rho': 1.3270738739436174, 'eta': 0.039895395830785316, 'T': 1.028726511812752, 'beta': 0.757405041715233, 'nf_rls': 6, 'ff_rls': 0.956649459266282, 'dt_rls': 319558.18917694926}
Trial #138 [COMPLETE] | Score: 0.0175 | Params: {'rho': 0.1856